# 05C - Cinematic Piano Memory

**Guiding question:** can you hear, see, and measure whether a recurrent model keeps an opening key cue across an entire phrase?

> **What you finished last time:** 05B established explicit PyTorch sequence, gradient, and optimization contracts.
> **What this notebook delivers:** an original D-minor piano phrase, matched RNN/LSTM training, audible checkpoints, and a learner-controlled hidden-size comparison.
> **What this chapter is not:** a new API lesson; it reuses `sequence_helpers.py`.

The repeating clock never reveals absolute pitch. The model sees the key cue once, then must carry it.


In [ ]:
# -- Setup: reuse the canonical sequence helpers ---------------------------------
import io
import wave
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from sequence_helpers import backward_step, seed_everything, trainable_parameter_count

seed_everything(17)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.facecolor": "#1a1a2e", "axes.facecolor": "#1a1a2e"})
print(f"Device: {device}")


## 1 - Build one original phrase and make its dependency unfair

```mermaid
flowchart LR
    A["Opening D3 key cue"] --> B["Repeating step clock"]
    B --> C["24-note arpeggio"]
    C --> D["Final pitch error"]
    D --> E["Gradient back to opening cue"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Original D-minor progression and lightweight piano synthesizer --------------
midi_notes = np.array([
    50, 57, 62, 65, 69, 65,
    46, 53, 58, 62, 65, 62,
    41, 48, 53, 57, 60, 57,
    48, 55, 60, 64, 67, 62,
], dtype=np.float32)
normalized_target = torch.tensor((midi_notes - 50.0) / 12.0, dtype=torch.float32)

def synthesize_piano(midi, seconds_per_note=0.16, sample_rate=16000):
    chunks = []
    for note in midi:
        t = np.linspace(0, seconds_per_note, int(sample_rate * seconds_per_note), endpoint=False)
        frequency = 440.0 * 2.0 ** ((float(note) - 69.0) / 12.0)
        envelope = np.exp(-5.5 * t)
        tone = (
            np.sin(2 * np.pi * frequency * t)
            + 0.35 * np.sin(2 * np.pi * 2 * frequency * t)
            + 0.12 * np.sin(2 * np.pi * 3 * frequency * t)
        ) * envelope
        chunks.append(tone)
    audio = np.concatenate(chunks)
    audio = 0.85 * audio / max(np.abs(audio).max(), 1e-8)
    return audio.astype(np.float32), sample_rate

target_audio, sample_rate = synthesize_piano(midi_notes)
print(f"Phrase length: {len(midi_notes)} notes")
print("The key cue appears in the first feature only once.")
display(Audio(target_audio, rate=sample_rate))


In [ ]:
# -- Sequence features: key once, clock always, no absolute pitch after step zero -
steps = torch.arange(len(midi_notes), dtype=torch.float32)
phase = 2 * torch.pi * steps / 6
features = torch.stack(
    [
        torch.sin(phase),
        torch.cos(phase),
        (steps == 0).float(),
    ],
    dim=-1,
).unsqueeze(0)
features[0, 0, 2] = normalized_target[0]
targets = normalized_target.view(1, -1, 1)

print(f"Features: {tuple(features.shape)} = (batch, time, feature)")
print(f"Targets:  {tuple(targets.shape)}")
print("After timestep zero, no input feature contains absolute pitch.")


## 2 - Match everything except the memory cell

```mermaid
flowchart LR
    X["Same 3 features"] --> R["Vanilla nn.RNN"]
    X --> L["nn.LSTM"]
    R --> HR["Same linear pitch head"]
    L --> HL["Same linear pitch head"]
    HR --> M["Same MSE and optimizer"]
    HL --> M
    style X fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style L fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style HR fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style HL fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Transparent model wrapper: the cell type is the controlled variable --------
class PianoMemoryModel(nn.Module):
    def __init__(self, cell_type, hidden_size=24):
        super().__init__()
        recurrent_class = nn.RNN if cell_type == "rnn" else nn.LSTM
        self.recurrent = recurrent_class(3, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, sequence):
        hidden_path, _ = self.recurrent(sequence)
        return self.head(hidden_path), hidden_path

def train_model(cell_type, hidden_size=24, epochs=500, snapshot_epochs=(0, 50, 150, 499)):
    seed_everything(17)
    model = PianoMemoryModel(cell_type, hidden_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
    snapshots = {}
    losses = []
    norms = []

    for epoch in range(epochs):
        predictions, _ = model(features.to(device))
        loss = nn.functional.mse_loss(predictions, targets.to(device))
        pre_clip, post_clip = backward_step(loss, model, optimizer, clip_norm=1.0)
        losses.append(float(loss.detach()))
        norms.append((pre_clip, post_clip))
        if epoch in snapshot_epochs:
            snapshots[epoch] = predictions.detach().cpu().squeeze().numpy()
    return model, losses, norms, snapshots

rnn_model, rnn_losses, rnn_norms, rnn_snapshots = train_model("rnn")
lstm_model, lstm_losses, lstm_norms, lstm_snapshots = train_model("lstm")
print(f"RNN parameters:  {trainable_parameter_count(rnn_model):,}")
print(f"LSTM parameters: {trainable_parameter_count(lstm_model):,}")
print(f"Final MSE - RNN: {rnn_losses[-1]:.6f}, LSTM: {lstm_losses[-1]:.6f}")


In [ ]:
# -- Visual payoff: learning curves and final pitch paths -------------------------
with torch.inference_mode():
    rnn_prediction, _ = rnn_model(features.to(device))
    lstm_prediction, _ = lstm_model(features.to(device))
rnn_midi = rnn_prediction.cpu().squeeze().numpy() * 12 + 50
lstm_midi = lstm_prediction.cpu().squeeze().numpy() * 12 + 50

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].semilogy(rnn_losses, label="RNN", color="#f97316")
axes[0].semilogy(lstm_losses, label="LSTM", color="#60a5fa")
axes[0].set(title="Matched training", xlabel="Epoch", ylabel="MSE")
axes[0].legend()
axes[1].plot(midi_notes, label="target", color="#34d399", linewidth=3)
axes[1].plot(rnn_midi, label="RNN", color="#f97316")
axes[1].plot(lstm_midi, label="LSTM", color="#60a5fa")
axes[1].set(title="Final pitch path", xlabel="Timestep", ylabel="MIDI note")
axes[1].legend()
plt.show()


## 3 - Hear checkpointed practice

The audio is generated from recorded model predictions, not a hand-drawn success curve.


In [ ]:
# -- Audible checkpoints from actual saved predictions ---------------------------
for epoch in sorted(lstm_snapshots):
    checkpoint_midi = lstm_snapshots[epoch] * 12 + 50
    audio, rate = synthesize_piano(checkpoint_midi)
    print(f"LSTM checkpoint epoch {epoch}, MSE={lstm_losses[epoch]:.6f}")
    display(Audio(audio, rate=rate))


## 4 - Trace the final mistake back to the opening cue

**Predict:** which architecture sends a larger gradient from the final pitch error to the first input cue?


In [ ]:
# -- Measured memory path: final loss to opening input ----------------------------
def opening_cue_gradient(model):
    probe = features.clone().to(device).requires_grad_(True)
    prediction, _ = model(probe)
    final_loss = (prediction[:, -1] - targets.to(device)[:, -1]).pow(2).mean()
    final_loss.backward()
    return float(probe.grad[0, 0].norm()), float(final_loss.detach())

rnn_cue_gradient, rnn_final_error = opening_cue_gradient(rnn_model)
lstm_cue_gradient, lstm_final_error = opening_cue_gradient(lstm_model)
print(f"RNN  opening-cue gradient: {rnn_cue_gradient:.8f}; final error={rnn_final_error:.8f}")
print(f"LSTM opening-cue gradient: {lstm_cue_gradient:.8f}; final error={lstm_final_error:.8f}")
print("Interpret the gradient with the error: a tiny gradient is useful only if the final error is also tiny.")


## 5 - Mandatory learner comparison

**Your turn:** change only the hidden size. Predict whether extra capacity improves the LSTM's final MSE enough to justify its parameter cost.


In [ ]:
# -- Your turn: one controlled hidden-size comparison ----------------------------
# CHANGE THIS: try 12, 24, or 36
hidden_size_to_test = 12
test_model, test_losses, _, _ = train_model(
    "lstm",
    hidden_size=hidden_size_to_test,
    epochs=250,
    snapshot_epochs=(),
)
print(f"Hidden size {hidden_size_to_test}: {trainable_parameter_count(test_model):,} parameters")
print(f"Final 250-epoch MSE: {test_losses[-1]:.6f}")
print("Check: compare both error and parameter count; width is not free.")


In [ ]:
# -- Final inline players: authored target, RNN, and LSTM -------------------------
for label, notes in [
    ("Authored target", midi_notes),
    ("Vanilla RNN", rnn_midi),
    ("LSTM", lstm_midi),
]:
    audio, rate = synthesize_piano(notes)
    print(label)
    display(Audio(audio, rate=rate))


## 6 - Evidence scorecard

| Question | Evidence |
|---|---|
| Is the task actually long-range? | absolute pitch appears only in the opening cue |
| Is the comparison controlled? | same inputs, head, loss, optimizer, and clipping |
| Is the payoff real? | plots and audio come from saved predictions |
| Can the final error teach the opening? | input-gradient trace reports the route |
| Did the learner vary one factor? | hidden-size comparison reports error and cost |

**Built and measured:** original music task, matched training, audible checkpoints, gradient path, width trade-off.

**Explained or illustrated:** why final error and gradient magnitude must be interpreted together.

**Named and out of scope:** polyphonic event tokenization, sustain-pedal modeling, and attention.

You can now feel the residual complaint: even a good recurrent memory passes information through every intermediate step. The Transformer route will create direct position-to-position connections instead.
